# Hands-on with Kafka

![kafka](https://images.contentful.com/gt6dp23g0g38/53UO4964r0e7kRVm0mcUUZ/f6f6d7b1b90e8e88a5be0d1845bdf950/what_is_kafka_and_how_does_it_work.png)

## Installing Kafka Python Client

First, we'll need to install Kafka Python client `kafka-python` to consume messages from a Kafka cluster.

In [1]:
import json
from kafka import KafkaConsumer, TopicPartition
from dotenv import load_dotenv
import os
from pprint import pprint
import ssl

In [2]:
load_dotenv()

AIVEN_HOST = os.getenv("AIVEN_HOST")
AIVEN_PORT = os.getenv("AIVEN_PORT")
BOOTSTRAP_SERVERS = f"{AIVEN_HOST}:{AIVEN_PORT}"
AIVEN_USERNAME = os.getenv("AIVEN_USERNAME")
AIVEN_PASSWORD = os.getenv("AIVEN_PASSWORD")


CONSUMER_GROUP = 'pizza-consumer-group-test3'

First, we'll instantiate the consumer.

Our data was serialized in JSON during publishing, hence we'll need to deserialize as JSON when consuming. In actual production, it is recommended to use the binary format that we've learnt, such as Avro.

In [3]:
ssl_context = ssl.create_default_context(
    cafile="../ca.pem"
)

consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    security_protocol='SASL_SSL',
    sasl_mechanism='SCRAM-SHA-256',
    sasl_plain_username=AIVEN_USERNAME,
    sasl_plain_password=AIVEN_PASSWORD,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    key_deserializer=lambda v: json.loads(v.decode('utf-8')) if v is not None else None,
    ssl_context=ssl_context,
    group_id=CONSUMER_GROUP,
)
print(f"Consumer created")

Consumer created


We set the offset for the consumer to be `earliest` to get the earliest order (message).

List the available topics:

In [4]:
consumer.topics()

{'_schemas', 'pizza-workflow', 'user_activity_data_gen'}

Let's assume we are the owners of a pizza delivery chain, and we'd like to push the orders to Apache Kafka as they come in.

We have a topic consisting of pizza orders, which contains the Order ID, Order Details (pizzas and toppings), client's Name, Address, Phone Number, the Shop Name, Total Cost and Timestamp.

There is only one partition for the topic. First, let's subscribe to the topic:

In [9]:
topic_name = "pizza-workflow"

In [10]:
consumer.subscribe(topics=[topic_name])
consumer.subscription()

{'pizza-workflow'}

We can then start reading the events, let's read and (pretty) print the first 5:

In [11]:
from pprint import pprint

In [12]:
for ix, message in enumerate(consumer, start=1):
    print("%d:%d: k=%s" % (message.partition,
                           message.offset,
                           message.key))
    pprint("-------------------")
    pprint(message.value)
    print()
    if ix == 5:
        break

0:0: k={'shop': 'Mammamia Pizza'}
'-------------------'
{'address': '21087 Calvin Plains\nJonesland, NY 76392',
 'cost': 41.93,
 'id': 0,
 'name': 'Jessica Smith',
 'phoneNumber': '001-701-915-3000',
 'pizzas': [{'additionalToppings': ['🌶️ hot pepper', '🐟 tuna'],
             'pizzaName': 'Salami'},
            {'additionalToppings': [], 'pizzaName': 'Mari & Monti'},
            {'additionalToppings': ['🌶️ hot pepper'], 'pizzaName': 'Salami'},
            {'additionalToppings': ['🐟 tuna'], 'pizzaName': 'Peperoni'}],
 'shop': 'Mammamia Pizza',
 'timestamp': 1781236491437}

0:1: k={'shop': 'Mammamia Pizza'}
'-------------------'
{'address': '19721 Drew Key\nNew Donaldport, NH 05690',
 'cost': 7.74,
 'id': 1,
 'name': 'Roger Brown',
 'phoneNumber': '475-943-3780x8105',
 'pizzas': [{'additionalToppings': ['🍓 strawberry'], 'pizzaName': 'Marinara'},
            {'additionalToppings': ['🌶️ hot pepper',
                                    '🧀 blue cheese',
                                    '🍍

Now, continue printing the next 5 events, notice the offset number and order id:

In [14]:
for ix, message in enumerate(consumer, start=1):
    print("%d:%d: k=%s" % (message.partition,
                           message.offset,
                           message.key))
    pprint(message.value)
    print()
    if ix == 5:
        break

0:0: k={'shop': 'Mammamia Pizza'}
{'address': '21087 Calvin Plains\nJonesland, NY 76392',
 'cost': 41.93,
 'id': 0,
 'name': 'Jessica Smith',
 'phoneNumber': '001-701-915-3000',
 'pizzas': [{'additionalToppings': ['🌶️ hot pepper', '🐟 tuna'],
             'pizzaName': 'Salami'},
            {'additionalToppings': [], 'pizzaName': 'Mari & Monti'},
            {'additionalToppings': ['🌶️ hot pepper'], 'pizzaName': 'Salami'},
            {'additionalToppings': ['🐟 tuna'], 'pizzaName': 'Peperoni'}],
 'shop': 'Mammamia Pizza',
 'timestamp': 1781236491437}

0:1: k={'shop': 'Mammamia Pizza'}
{'address': '19721 Drew Key\nNew Donaldport, NH 05690',
 'cost': 7.74,
 'id': 1,
 'name': 'Roger Brown',
 'phoneNumber': '475-943-3780x8105',
 'pizzas': [{'additionalToppings': ['🍓 strawberry'], 'pizzaName': 'Marinara'},
            {'additionalToppings': ['🌶️ hot pepper',
                                    '🧀 blue cheese',
                                    '🍍 pineapple'],
             'pizzaName': 'Mar

In [13]:
# go to beginning 
consumer.seek_to_beginning()

In [15]:
# close consumer 
consumer.close()